# Stage 2 — Subtask 7: Data Quality Verification

**Run this after splits are created (ST4) and before any training.**

This notebook:
1. Cross-split near-duplicate detection (pHash) — prevents accuracy inflation
2. Cross-dataset contamination check (PlantVillage vs PlantDoc)
3. SSCD alternative — documented for high-precision cases
4. CleanLab execution — if Stage 3 probabilities are available
5. Issues an **integrity certificate** — training scripts should check this

> If the certificate shows `"status": "ISSUES_FOUND"`, resolve before starting Stage 3.


In [ ]:
%pip install imagehash --quiet


In [2]:
import json, sys
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from config_stage2 import *


QUALITY_OUT.mkdir(parents=True, exist_ok=True)

print("Setup complete.")


Setup complete.


## 1. pHash Deduplication Functions

In [6]:
import imagehash

def compute_hashes(paths: list, desc: str) -> dict:
    hashes = {}
    for p in tqdm(paths, desc=desc, leave=False):
        try:
            with Image.open(p) as img:
                hashes[str(p)] = imagehash.phash(img.convert("L"), hash_size=HASH_SIZE)
        except Exception:
            pass
    return hashes

def find_duplicates(paths_a, paths_b, label_a, label_b, threshold=PHASH_THRESHOLD) -> dict:
    if not paths_a or not paths_b:
        return {"status": "SKIPPED — paths not found"}
    ha = compute_hashes(paths_a, f"Hash {label_a}")
    hb = compute_hashes(paths_b, f"Hash {label_b}")

    pairs = []
    for pa, hha in tqdm(ha.items(), desc="Comparing", leave=False):
        for pb, hhb in hb.items():
            d = hha - hhb
            if d <= threshold:
                pairs.append({"path_a": pa, "path_b": pb, "hamming": int(d),
                              "class_a": Path(pa).parent.name, "class_b": Path(pb).parent.name})

    rate = len(pairs) / max(len(ha), 1) * 100
    sev  = ("CRITICAL" if rate > 5 else "WARNING" if rate > 1 else "LOW" if rate > 0 else "CLEAN")

    icon = "✅" if sev == "CLEAN" else "⚠️"
    print(f"\n{icon} [{label_a} vs {label_b}]")
    print(f"   Duplicate pairs : {len(pairs)}")
    print(f"   Contamination   : {rate:.3f}%  [{sev}]")
    if pairs:
        print(f"   ACTION REQUIRED : Remove flagged images from training split")

    return {"comparison": f"{label_a} vs {label_b}", "pairs": len(pairs),
            "contamination_pct": round(rate, 4), "severity": sev,
            "action_required": bool(pairs), "pairs_sample": pairs[:20]}


## 2. Train vs Test Check

**Most critical check.** Any near-duplicate in both train and test directly inflates reported accuracy.


In [7]:
all_checks = {}

split_train = SPLITS_DIR / "train_split.json"
split_test  = SPLITS_DIR / "test_split.json"

if split_train.exists() and split_test.exists():
    train_data = json.loads(split_train.read_text())
    test_data  = json.loads(split_test.read_text())
    np.random.seed(RANDOM_SEED)
    tr_sample = list(np.random.choice(train_data["paths"], min(3000, len(train_data["paths"])), replace=False))
    te_sample = list(np.random.choice(test_data["paths"],  min(1000, len(test_data["paths"])),  replace=False))
    check1 = find_duplicates([Path(p) for p in tr_sample], [Path(p) for p in te_sample],
                              "train", "test")
else:
    print("⚠️  Splits not found — run 06_label_splits.ipynb first")
    check1 = {"status": "PENDING"}

all_checks["train_vs_test"] = check1



✅ [train vs test]
   Duplicate pairs : 0
   Contamination   : 0.000%  [CLEAN]


## 3. PlantVillage vs PlantDoc Cross-Dataset Check

Verifies no PlantDoc evaluation image appeared in PlantVillage training data.


In [8]:
if PLANTVILLAGE_RAW_DIR.exists() and PLANTDOC_RAW_DIR.exists():
    np.random.seed(RANDOM_SEED)
    pv_paths = [p for p in PLANTVILLAGE_RAW_DIR.rglob("*") if p.suffix.lower() in VALID_EXT]
    pd_paths = [p for p in PLANTDOC_RAW_DIR.rglob("*") if p.suffix.lower() in VALID_EXT]
    pv_sample = list(np.random.choice(pv_paths, min(2000, len(pv_paths)), replace=False))
    check2 = find_duplicates(pv_sample, pd_paths, "PlantVillage", "PlantDoc")
else:
    print("⚠️  Dataset directories not found — skipping cross-dataset check")
    check2 = {"status": "PENDING — datasets not found at configured paths"}

all_checks["plantvillage_vs_plantdoc"] = check2



✅ [PlantVillage vs PlantDoc]
   Duplicate pairs : 0
   Contamination   : 0.000%  [CLEAN]


## 4. SSCD — High-Precision Alternative

SSCD (Self-Supervised Copy Detection, Meta AI 2022) catches semantically similar near-duplicates that pHash misses. Use if pHash detects > 0.5% contamination or you want higher confidence claims.


In [9]:
sscd_doc = {
    "method": "SSCD — Self-Supervised Copy Detection",
    "citation": "Pizzi et al. (2022). A Self-Supervised Descriptor for Image Copy Detection. CVPR.",
    "when_to_use": "If pHash detects >0.5% contamination or false positives suspected.",
    "advantage_over_phash": (
        "pHash is pixel-similarity based. SSCD uses learned copy embeddings: "
        "correctly distinguishes (a) two different images of the same disease [NOT duplicates] "
        "from (b) two crops of the same physical image [DUPLICATES]."
    ),
    "workflow": [
        "1. Download: resnet50_hard_mining_0.1.torchscript.pt from Meta AI GitHub",
        "2. Extract 512-d copy embeddings for all training + test images",
        "3. Build FAISS index over training embeddings",
        "4. Query each test image; flag cosine similarity > 0.9 as likely copy",
        "5. Manual review of flagged pairs before removal",
    ],
}
all_checks["sscd_alternative"] = sscd_doc
print("SSCD alternative documented.")


SSCD alternative documented.


## 5. CleanLab Label Noise Estimation

Requires Stage 3 baseline model probabilities. Run this cell again after Stage 3 is complete.


In [10]:
prob_path  = OUTPUTS_DIR / "baseline" / "oof_probabilities.npy"
label_path = OUTPUTS_DIR / "baseline" / "oof_labels.npy"

if prob_path.exists() and label_path.exists():
    try:
        from cleanlab.filter import find_label_issues
        probs  = __import__("numpy").load(str(prob_path))
        labels = __import__("numpy").load(str(label_path))
        issues = find_label_issues(labels=labels, pred_probs=probs,
                                   return_indices_ranked_by="self_confidence")
        noise_rate = len(issues) / len(labels) * 100
        print(f"CleanLab: {len(issues)} likely mislabelled ({noise_rate:.2f}% noise rate)")
        check_cl = {"status": "COMPLETE", "flagged": len(issues), "noise_pct": round(noise_rate,3)}
    except Exception as e:
        check_cl = {"status": f"ERROR: {e}"}
else:
    print("⏳ CleanLab pending — waiting for Stage 3 baseline model probabilities")
    print("   Required files:")
    print(f"     {prob_path}")
    print(f"     {label_path}")
    check_cl = {
        "status": "PENDING — requires Stage 3",
        "action": (
            "After ResNet-50 baseline training: save out-of-fold softmax probabilities "
            "as oof_probabilities.npy (shape: N_train × N_classes) and oof_labels.npy, "
            "then re-run this cell."
        ),
    }

all_checks["cleanlab"] = check_cl


⏳ CleanLab pending — waiting for Stage 3 baseline model probabilities
   Required files:
     D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\baseline\oof_probabilities.npy
     D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\baseline\oof_labels.npy


## 6. Integrity Certificate

In [11]:
# Generate certificate
issues = [c for c in all_checks.values()
          if isinstance(c, dict) and c.get("action_required", False)]

certificate = {
    "project": "Leaf-JEPA",
    "stage": "2 — Dataset Preparation",
    "checks": list(all_checks.keys()),
    "status": "PASSED" if not issues else "ISSUES_FOUND",
    "issues_requiring_action": len(issues),
    "message": (
        "✅ All checks passed. Stage 3 may begin."
        if not issues else
        f"⚠️  {len(issues)} issue(s) require resolution before Stage 3."
    ),
    "check_details": all_checks,
}

cert_path = QUALITY_OUT / "integrity_certificate.json"
with open(cert_path, "w") as f:
    json.dump(certificate, f, indent=2, default=str)

print(f"\n{'='*55}")
print(f"INTEGRITY CERTIFICATE: {certificate['status']}")
print(certificate["message"])
print(f"Certificate → {cert_path}")
print(f"{'='*55}")
print("\n✅ SUBTASK 7 COMPLETE")



INTEGRITY CERTIFICATE: PASSED
✅ All checks passed. Stage 3 may begin.
Certificate → D:\IRP\leaf-jepa\stage2_dataset_preparation\outputs\quality_verification\integrity_certificate.json

✅ SUBTASK 7 COMPLETE
